# Pawikan Sentinel - Model Training & Optimization Notebook

This notebook automates the process of training a YOLOv5 model for sea turtle detection, optimizing it for deployment on a Raspberry Pi, and saving the artifacts to Google Drive.

**Notebook Structure:**
1.  **Setup:** Mounts Google Drive, clones the YOLOv5 repo, and installs dependencies.
2.  **Data Preparation:** Copies the `pawikan_dataset.zip` from Google Drive to the Colab environment and unzips it.
3.  **Training:** Automatically resumes the last training run or starts a new one if none exists. Saves all checkpoints and results directly to Google Drive to prevent data loss.
4.  **Model Optimization:** Takes the best model from the training run (`best.pt`) and uses YOLOv5's integrated `export.py` script to convert it to ONNX and TensorFlow Lite (with INT8 quantization) for efficient inference.
5.  **Save Artifacts:** Copies the final `.onnx` and `.tflite` models and all training logs to designated folders in your Google Drive.

In [ ]:
# Cell 1: Setup
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define project path in Google Drive and local path for YOLOv5
project_path = "/content/drive/MyDrive/PawikanSentinel"
yolo_path = "/content/yolov5"

# Clone YOLOv5 repository if it doesn't exist
if not os.path.exists(yolo_path):
    print("Cloning YOLOv5 repository...")
    !git clone https://github.com/ultralytics/yolov5 {yolo_path}
else:
    print("YOLOv5 repository already exists.")

# Install dependencies from the YOLOv5 requirements file
%cd {yolo_path}
!pip install -r requirements.txt
%cd /content/

In [ ]:
# Cell 2: Data Preparation
import zipfile
import os

# Define paths for the dataset
dataset_zip_path = os.path.join(project_path, "datasets", "pawikan_dataset.zip")
colab_dataset_path = "/content/datasets"

# Create directory for the dataset in the Colab environment
if not os.path.exists(colab_dataset_path):
    os.makedirs(colab_dataset_path)

# Copy the dataset from Google Drive to the Colab instance
print(f"Copying dataset from {dataset_zip_path}...")
!cp {dataset_zip_path} {colab_dataset_path}/pawikan_dataset.zip

# Unzip the dataset
print("Unzipping dataset...")
with zipfile.ZipFile(os.path.join(colab_dataset_path, "pawikan_dataset.zip"), 'r') as zip_ref:
    zip_ref.extractall(colab_dataset_path)

print("Dataset prepared.")
# Verify the structure of the unzipped dataset
!ls -R {colab_dataset_path}

In [ ]:
# Cell 3: Training
import glob
import os

%cd {yolo_path}

# Define paths for training
drive_project_path = "/content/drive/MyDrive/PawikanSentinel"
dataset_yaml_path = "/content/datasets/pawikan_dataset/dataset.yaml"
runs_path = os.path.join(drive_project_path, "runs", "train")

# Check if a previous training run exists in Google Drive
exp_paths = sorted(glob.glob(os.path.join(runs_path, "exp*")))

if exp_paths:
    print("Previous training run found. Resuming training.")
    # The --resume flag tells train.py to find the last run in the --project dir and continue
    !python train.py --resume --project {runs_path}
else:
    print("No previous training run found. Starting a new training.")
    # --name 'exp' ensures the first run is created as 'exp', subsequent resumes will continue in it
    !python train.py --img 640 --batch 16 --epochs 100 --data {dataset_yaml_path} --weights yolov5n.pt --project {runs_path} --name exp

### 4. Model Optimization using `export.py`

After training, the model (saved as `best.pt`) is in the PyTorch format. For deployment on edge devices like the Raspberry Pi, we need to convert it to a more efficient format. We use YOLOv5's built-in `export.py` script for this process.

**Key Steps:**
1.  **Find the Best Model:** The script first locates the `best.pt` file from the most recent training experiment.
2.  **Convert to ONNX:** The model is first converted to the Open Neural Network Exchange (ONNX) format. This is a common intermediate step that decouples the model from its original framework (PyTorch).
3.  **Convert to TensorFlow Lite:** The ONNX model is then converted to a TensorFlow Lite (`.tflite`) model. TFLite is designed specifically for deploying models on mobile and embedded devices.
4.  **Apply INT8 Quantization:** During the TFLite conversion, we use `--int8` quantization. This process converts the model's 32-bit floating-point weights to 8-bit integers. This dramatically reduces the model size (by ~4x) and speeds up inference on compatible hardware, with a minimal trade-off in accuracy. The `--data` flag is required for INT8 quantization, as it uses the training data to determine the best way to quantize the model's weights and activations.

In [ ]:
# Cell 5: Model Optimization
import glob
import os

# Define paths
runs_path = os.path.join(project_path, "runs", "train")
dataset_yaml_path = os.path.join(colab_dataset_path, "pawikan_dataset", "dataset.yaml")

# Find the latest training run directory
list_of_exps = glob.glob(os.path.join(runs_path, "exp*"))
if not list_of_exps:
    raise Exception("No training runs found. Please run the training cell first.")
latest_exp = max(list_of_exps, key=os.path.getctime)
print(f"Latest training run found at: {latest_exp}")

# Path to the best weights from the training
best_weights_path = os.path.join(latest_exp, "weights", "best.pt")
print(f"Best weights found at: {best_weights_path}")

# --- Export to ONNX and TFLite ---
%cd {yolo_path}

# Execute the export script with comprehensive arguments:
# --weights: Path to the trained PyTorch model (.pt file) to be converted.
# --include: Specifies the output formats. Here, we want 'onnx' and 'tflite'.
# --int8: Enables INT8 (8-bit integer) quantization for the TFLite model.
#         This significantly reduces model size and speeds up inference.
# --data: Path to the dataset's .yaml file. This is crucial for INT8 quantization
#         as it provides the calibration data needed to accurately convert weights
#         from float32 to int8 without a major loss in accuracy.
!python export.py --weights {best_weights_path} --include onnx tflite --int8 --data {dataset_yaml_path}

# Define paths to the exported models. The export script appends suffixes automatically.
onnx_model_path = best_weights_path.replace(".pt", ".onnx")
tflite_model_path = best_weights_path.replace(".pt", "-int8.tflite")

print(f"
ONNX model saved at: {onnx_model_path}")
print(f"TFLite INT8 model saved at: {tflite_model_path}")

In [ ]:
# Cell 6: Save Artifacts
import os
import shutil

# Define destination directories on Google Drive
final_models_dir = os.path.join(project_path, "models")
logs_dir = os.path.join(project_path, "logs")
os.makedirs(final_models_dir, exist_ok=True)
os.makedirs(logs_dir, exist_ok=True)

# --- Copy final models ---
if os.path.exists(onnx_model_path):
    shutil.copy(onnx_model_path, final_models_dir)
    print(f"Copied {os.path.basename(onnx_model_path)} to {final_models_dir}")

if os.path.exists(tflite_model_path):
    shutil.copy(tflite_model_path, final_models_dir)
    print(f"Copied {os.path.basename(tflite_model_path)} to {final_models_dir}")

# --- Copy training logs ---
# The entire run directory for the latest experiment contains all logs, plots, etc.
exp_name = os.path.basename(latest_exp)
log_destination = os.path.join(logs_dir, exp_name)

# Remove old log backup to copy the new one, preventing errors
if os.path.exists(log_destination):
    shutil.rmtree(log_destination)
shutil.copytree(latest_exp, log_destination)

print(f"Copied latest training run logs from {latest_exp} to {log_destination}")
print("
All artifacts saved to Google Drive.")